<a href="https://colab.research.google.com/github/vnm19102-hub/Gen-AI-Assignments/blob/main/CRUD_Ope_chromaDB_41229.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

In [ ]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Create a fresh collection to practice CRUD operations on
collection = client.create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: crud_demo
Current document count: 0


In [ ]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 39.8MiB/s]


Documents added! Total count: 5


In [ ]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

doc1 -> Python is a popular programming language for AI.
doc2 -> The weather today is sunny and warm.
doc3 -> Machine learning models learn patterns from data.


In [ ]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

doc1 | {'category': 'tech'} | Python is a popular programming language for AI.
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


In [ ]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc1 | distance=0.9516 | Python is a popular programming language for AI.


In [ ]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc4 | distance=1.3895 | I enjoy playing football on weekends.


In [ ]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

['I enjoy playing football and basketball on weekends.']
[{'category': 'sports', 'updated': True}]


In [ ]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",  # existing id -> will update
        "The Eiffel Tower is located in Paris."                           # new id -> will create
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

Total count after upsert: 6
['Deep learning uses neural networks with many layers and GPUs.', 'The Eiffel Tower is located in Paris.']


In [ ]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['doc1', 'doc3', 'doc4', 'doc5', 'doc6']


In [ ]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 2
Remaining ids: ['doc4', 'doc6']


In [ ]:
final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 2
doc4 | {'category': 'sports', 'updated': True} | I enjoy playing football and basketball on weekends.
doc6 | None | The Eiffel Tower is located in Paris.


# **Practice 1 - adding own documents**

In [ ]:
# Add 5 new documents

collection.add(
    documents=[
        "Python is widely used for data science.",
        "Artificial intelligence enables machines to perform intelligent tasks.",
        "SQL is used to manage and query relational databases.",
        "Football is one of the most popular sports in the world.",
        "Cloud computing provides computing resources over the internet."
    ],
    metadatas=[
        {"category": "programming"},
        {"category": "ai"},
        {"category": "database"},
        {"category": "sports"},
        {"category": "cloud"}
    ],
    ids=["practice1", "practice2", "practice3", "practice4", "practice5"]
)

print("Documents added!")
print("Total count:", collection.count())

Documents added!
Total count: 7


In [ ]:
# Verify the updated and newly created documents

result = collection.get(
    ids=["practice1", "practice2", "practice6"]
)

for doc_id, doc, meta in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

practice1 | {'category': 'programming'} | Python is widely used for data science.
practice2 | {'category': 'ai'} | Artificial intelligence enables machines to perform intelligent tasks.


# **Practice 2 - Deleting Doc , using Category**

In [ ]:
# Delete all documents where category is "sports"

collection.delete(
    where={"category": "sports"}
)

# Confirm the remaining document count
print("Total count after deleting sports documents:", collection.count())

Total count after deleting sports documents: 5


In [ ]:
# Display remaining documents

remaining = collection.get()

print("Remaining IDs:", remaining["ids"])

for doc_id, doc, meta in zip(
    remaining["ids"],
    remaining["documents"],
    remaining["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

Remaining IDs: ['doc6', 'practice1', 'practice2', 'practice3', 'practice5']
doc6 | None | The Eiffel Tower is located in Paris.
practice1 | {'category': 'programming'} | Python is widely used for data science.
practice2 | {'category': 'ai'} | Artificial intelligence enables machines to perform intelligent tasks.
practice3 | {'category': 'database'} | SQL is used to manage and query relational databases.
practice5 | {'category': 'cloud'} | Cloud computing provides computing resources over the internet.


# **Practice 3 - using upsert() function**

In [ ]:
def safe_add(collection, id, text):
    collection.upsert(
        ids=[id],
        documents=[text]
    )

    print(f"Document '{id}' added or updated successfully.")

In [ ]:
safe_add(
    collection,
    "doc7",
    "Natural language processing helps computers understand human language."
)

Document 'doc7' added or updated successfully.


In [ ]:
safe_add(
    collection,
    "doc7",
    "Natural language processing is widely used in modern AI applications."
)

Document 'doc7' added or updated successfully.
